# CS5487 Digits Project In Colab

Use this notebook as the primary execution surface when local CPU and memory are limited. It reuses the Python package in this project instead of duplicating the experiment logic.

In [2]:
USE_GOOGLE_DRIVE = True
PROJECT_ROOT = (
    "/content/drive/MyDrive/CS5487 Course Project Code"
    if USE_GOOGLE_DRIVE
    else "/content/CS5487 Course Project Code"
)
GRID_SEARCH_JOBS = 1
# Use one of: None, "light", "heavy", "rbf_only", "random_forest_only", "mlp_only".
BATCH_PRESET = "heavy"
# Set False when you only want to inspect or combine finished batch runs.
RUN_EXPERIMENTS = True
SELECTED_TRIAL_NAMES = None
SELECTED_MODEL_NAMES = None
RUN_NAME = None
# Example: ["batch_light", "batch_heavy"]
COMBINE_RUN_NAMES = None

In [3]:
from pathlib import Path
import importlib

if USE_GOOGLE_DRIVE:
    colab_drive = importlib.import_module("google.colab.drive")
    colab_drive.mount("/content/drive")

project_root = Path(PROJECT_ROOT).expanduser()
print("Project root:", project_root)
if not project_root.exists():
    raise FileNotFoundError(
        "Project root not found. Update PROJECT_ROOT so it points to the folder "
        "that contains requirements.txt, src/, digits4000_txt/, and challenge/."
    )

Mounted at /content/drive
Project root: /content/drive/MyDrive/CS5487 Course Project Code


In [6]:
import os
import subprocess
import sys

os.chdir(project_root)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print('Working directory:', project_root)

Working directory: /content/drive/MyDrive/CS5487 Course Project Code


In [8]:
from digits_project.config import configure_project
from digits_project.experiment import combine_experiment_runs, run_project_experiments
from digits_project import config as project_config

# Import directly from submodules so Colab does not depend on __init__.py re-exports.
BATCH_PRESETS = {
    "light": {
        "run_name": "batch_light",
        "selected_model_names": [
            "knn_1",
            "logistic_regression_ova",
            "linear_svm_ova",
        ],
    },
    "heavy": {
        "run_name": "batch_heavy",
        "selected_model_names": [
            "rbf_svm_ova",
            "random_forest",
            "mlp",
        ],
    },
    "rbf_only": {
        "run_name": "batch_rbf_svm",
        "selected_model_names": ["rbf_svm_ova"],
    },
    "random_forest_only": {
        "run_name": "batch_random_forest",
        "selected_model_names": ["random_forest"],
    },
    "mlp_only": {
        "run_name": "batch_mlp",
        "selected_model_names": ["mlp"],
    },
}

if BATCH_PRESET is not None:
    if BATCH_PRESET not in BATCH_PRESETS:
        raise ValueError(f"Unknown BATCH_PRESET: {BATCH_PRESET}")
    RUN_NAME = BATCH_PRESETS[BATCH_PRESET]["run_name"]
    SELECTED_MODEL_NAMES = BATCH_PRESETS[BATCH_PRESET]["selected_model_names"]

configure_project(root_dir=project_root, grid_search_jobs=GRID_SEARCH_JOBS, run_name=RUN_NAME)
print(project_config.get_runtime_config())
print("Selected trials:", SELECTED_TRIAL_NAMES)
print("Selected models:", SELECTED_MODEL_NAMES)
print("Combine run names:", COMBINE_RUN_NAMES)

ImportError: cannot import name 'combine_experiment_runs' from 'digits_project.experiment' (/content/drive/MyDrive/CS5487 Course Project Code/src/digits_project/experiment.py)

In [ ]:
from digits_project.data import load_digits_project_data

bundle = load_digits_project_data()
print('digits4000 shape:', bundle.X.shape)
print('digits4000 labels:', bundle.y.shape)
print('challenge shape:', bundle.challenge_X.shape)
print('challenge labels:', bundle.challenge_y.shape)
print('trials:', [trial.name for trial in bundle.trials])

In [1]:
import pandas as pd
from digits_project.models import MODEL_SPECS

expected_pairs = {
    (trial.name, model_spec.name)
    for trial in bundle.trials
    for model_spec in MODEL_SPECS
}

results_paths = [project_root / "artifacts" / "results" / "final_selected_models.csv"]
runs_dir = project_root / "artifacts" / "runs"
if runs_dir.exists():
    results_paths.extend(sorted(runs_dir.glob("*/results/final_selected_models.csv")))

completed_pairs = set()
for result_path in results_paths:
    if not result_path.exists():
        continue
    frame = pd.read_csv(result_path)
    completed_pairs.update((row.trial, row.model) for row in frame.itertuples(index=False))
    print(f"{result_path.relative_to(project_root)} -> {len(frame)} rows")

print(f"Completed trial/model pairs: {len(completed_pairs)} / {len(expected_pairs)}")
pending_pairs = sorted(expected_pairs - completed_pairs)
if pending_pairs:
    print("Pending pairs:")
    print(pd.DataFrame(pending_pairs, columns=["trial", "model"]).to_string(index=False))
else:
    print("All trial/model pairs already have saved outputs.")

if runs_dir.exists():
    print("Available batch runs:", sorted(path.name for path in runs_dir.iterdir() if path.is_dir()))
else:
    print("Available batch runs: []")

ModuleNotFoundError: No module named 'digits_project'

In [ ]:
results = None
if RUN_EXPERIMENTS:
    results = run_project_experiments(
        root_dir=project_root,
        grid_search_jobs=GRID_SEARCH_JOBS,
        selected_trial_names=SELECTED_TRIAL_NAMES,
        selected_model_names=SELECTED_MODEL_NAMES,
        run_name=RUN_NAME,
    )
else:
    print("RUN_EXPERIMENTS is False, so this notebook will not start a new training run.")

In [ ]:
from pathlib import Path

if results is not None:
    print(results["summary"].to_string(index=False))
    print()
    print(results["email"].to_string(index=False))
    print()
    print("Run results saved under:", project_config.PATHS.results_dir)
else:
    print("No new experiment run was executed in this session.")

if COMBINE_RUN_NAMES:
    combined_results = combine_experiment_runs(root_dir=project_root, run_names=COMBINE_RUN_NAMES)
    print()
    print("Combined summary")
    print(combined_results["summary"].to_string(index=False))
    print()
    print("Combined results saved under:", Path(project_root) / "artifacts" / "results")
elif RUN_NAME is not None:
    print()
    print("Set COMBINE_RUN_NAMES to completed batch names and rerun this cell to rebuild artifacts/results.")
elif not RUN_EXPERIMENTS:
    print()
    print("Set COMBINE_RUN_NAMES to completed batch names if you only want to combine prior batch runs.")

## Challenge Reminder

Keep challenge accuracy out of presentation and poster material. The notebook can compute and save the private challenge results, but those numbers are only for the report and the instructor email workflow.